# Reproducing Fujimoto et al. (2026) from dandiset 001773

**Paper.** Fujimoto S, Fujimoto A, Elorette C, et al. (2026). Deep brain stimulation induces white
matter remodeling and functional changes to brain-wide networks. *Nature Neuroscience*.
[doi:10.1038/s41593-026-02301-4](https://doi.org/10.1038/s41593-026-02301-4). The claims file was
written from the version of record (open access, PMC13337501), with its Methods, Extended Data
legends and Supplementary Information.

**Dandiset.** [DANDI:001773](https://dandiarchive.org/dandiset/001773/0.260416.1939), version
0.260416.1939. This is the version the paper cites in its Code availability statement
(doi:10.48324/dandi.001773/0.260416.1939), as the claims file gives it. A later version,
0.260615.1838, holds the same 54 assets with the same asset ids (shown in the data check), so the
choice does not change the data.

The dandiset holds the paper's derived values as Excel sheets (voxel FA, voxel seed connectivity,
group connectomes, cell counts, g-ratios, behavior bins) and its MATLAB analysis scripts. The
sheets are read into memory from the archive; the analyses follow the scripts (see `dbs.py`). No
random numbers are used anywhere.

| Claim | Tested |
|---|---|
| ed1-behavior-increase | tested |
| ed1-monkey-d-no-change | tested |
| fig2a-cb-fa-increase | tested |
| fig2a-no-change-other-tracts | tested |
| ed2-cb-both-monkeys | tested |
| fig2b-mcc-cb-fa-increase | tested |
| fig3c-cc1-dapi-ratio | tested |
| ed5-cc1-count-dapi-unchanged | tested |
| ed6-dacc-ufm-no-oligodendrocyte-change | tested |
| fig3f-g-ratio-decrease | tested |
| fig4a-cortical-down-subcortical-up | tested |
| fig4b-roi-changes-counts | tested |
| fig4c-network-sorted-reduction | blocked: the connectome's 123 ROIs carry no names and no network assignment |
| fig4-nonsurgical-no-pattern | tested |
| fig5a-pre-scc-connectivity | tested |
| fig5b-scc-decreases | tested |
| fig5b-scc-increases | tested |
| ed9-control-hemisphere-changes | tested |
| ed9-no-change-motor-mcc | tested |
| ed7-decreases-both-monkeys | tested |
| fig5a-nonsurgical-no-reduction | blocked: no seed maps of the nonsurgical controls and no whole-brain voxel maps |
| fig5c-dmn-lim-cen-decrease | tested |
| fig5c-smn-increase | tested |
| fig5c-vis-no-change | tested |
| fig5c-san-rebalanced | tested |
| fig6a-monkey-d-cb-fa-decrease | tested |
| fig6b-monkey-d-dacc-cb-decrease | tested |
| fig6c-monkey-d-cc1-no-change | tested |
| ed10c-monkey-d-subcortical-decrease | tested |
| fig6d-monkey-d-scc-local-increase | blocked: no whole-brain voxel map of monkey D and no lead location |
| fig6e-monkey-d-no-network-reduction | tested, and blocked because its readings disagree |
| ed10d-monkey-d-roi-changes | tested |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import dbs

warnings.filterwarnings("ignore", category=UserWarning)       # openpyxl: unsupported Excel extensions
warnings.filterwarnings("ignore", category=RuntimeWarning)    # scipy quadrature at extreme tails
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)
started = time.time()
T = dbs.three_way_terms
LAB = dbs.roi_labels()
SIDE = {1: "stim", 2: "control"}
print(f"dandiset {dbs.DANDISET_ID}, version {dbs.DANDISET_VERSION}")

## Data check

The dandiset holds Excel sheets of derived values and the paper's MATLAB scripts; no images or
time series. Each check of the metadata against the values is stated under its output. With no
time series there is no clock to align; the corresponding check is that the condition labels mean
what the legends say. Side is checked against the hemispheres (voxel sheets with coordinates, the
NMT-space voxel counts, the EM image labels), and pre and post against the session dates of the
seed tables; pre and post cannot be told apart from the values themselves, so those labels are
taken as the sheets give them, which the reproduced statistics are consistent with.

In [ ]:
ids = dbs.asset_ids()
import requests
latest = requests.get(f"{dbs.API}/dandisets/{dbs.DANDISET_ID}/", timeout=60).json()["most_recent_published_version"]["version"]
url = f"{dbs.API}/dandisets/{dbs.DANDISET_ID}/versions/{latest}/assets/?page_size=200"
ids_latest = {a["path"]: a["asset_id"] for a in requests.get(url, timeout=60).json()["results"]}
print(f"{len(ids)} assets at {dbs.DANDISET_VERSION}; {len(ids_latest)} at the latest published version {latest}")
print("same paths and asset ids in both versions:", ids == ids_latest)
print(pd.Series([p.split("/")[0] for p in ids]).value_counts().to_string())
codes = {p: dbs.text(p) for p in ids if p.endswith(".m")}
seen = {}
for p, t in sorted(codes.items()):
    seen.setdefault(t, []).append(p.split("/")[-1])
print("\nScripts deposited more than once (identical text):")
for names in seen.values():
    if len(names) > 1:
        print("  ", names)

The two published versions hold the same 54 assets (same asset ids), so the choice of version
does not change the data. Two scripts are deposited twice under two names, with identical text;
each is counted once.

In [ ]:
b = dbs.behavior()
print("Behavior: 10-s bins per monkey (1 N, 2 T, 3 D), condition (stim 0 pre, 1 post) and group:")
print(b.groupby(["mk", "stim", "group"]).agg(bins=("sec", "size"), sec_min=("sec", "min"), sec_max=("sec", "max"),
      movement=("movement", "mean"), foraging=("foraging", "mean")).round(3).to_string())
print("values of movement:", sorted(b.movement.unique()), " foraging:", sorted(b.foraging.unique()))
legend = dbs.sheet("source_data/dbs_behavior_data_a.xlsx").iloc[1:6, 9:16]
print("\nThe sheet's own legend (columns J to P):")
print(legend.to_string(header=False, index=False))

Each monkey has 60 bins of 10 s (10 to 600 s) per session, as the Methods say. Every value of
both behaviors is 0 or 1. The sheet's legend codes foraging as yes 2 and no 1, which no value
uses; the values are read as for movement (1 yes, 0 no), which gives the Extended Data Fig. 1
statistics below. The legend gives group 1 for the DBS group and 0 for a non-stimulation control,
while monkey D's rows carry group 2. Both are recorded as metadata errors.

In [ ]:
fa = dbs.fa_wholetract()
print("FA whole tract, voxels per monkey (1 N, 2 T), side (1 stim, 2 control), tract (1 CB, 2 UF, 3 FM), pre and post:")
print(fa.groupby(["side", "tract", "mk", "prepost"]).size().unstack([2, 3]).to_string())
print("pooled per side and tract (pre):", fa[fa.prepost == 1].groupby(["side", "tract"]).size().to_dict())
print(f"FA range {fa.FA.min():.4f} to {fa.FA.max():.4f}; rows {len(fa)}")
sp = dbs.fa_subpart()
print("\nCB subregions (1 SCC, 2 dACC, 3 MCC, 4 PCC), pooled per side and part (pre):",
      sp[sp.prepost == 1].groupby(["side", "part"]).size().to_dict(), f"rows {len(sp)}, FA {sp.FA.min():.3f} to {sp.FA.max():.3f}")
faD, spD = dbs.fa_wholetract_mkD(), dbs.fa_subpart_mkD()
print("monkey D tracts (pre):", faD[faD.prepost == 1].groupby(["side", "tract"]).size().to_dict(), f"rows {len(faD)}")
print("monkey D CB subregions (pre):", spD[spD.prepost == 1].groupby(["side", "part"]).size().to_dict(), f"rows {len(spD)}")

The voxel counts match the legends of Fig. 2a (stimulated CB 380, UF 252, FM 105; control 363,
367, 133), Fig. 2b (stimulated SCC 54, dACC 137, MCC 55, PCC 80; control 81, 158, 50, 72), Fig. 6a
(sham CB 162, UF 120, FM 64; control 148, 74, 57) and Fig. 6b (sham 44, 58, 23, 16; control 28, 53,
13, 19), and the rows give the error df of the paper's models (3200, 1374, 1250, 508 values). The
pre and post values of a voxel are paired row by row (the same number of each per cell).

In [ ]:
# Which hemisphere is "side 1"? The per-tract voxel sheets (x, y, z, preFA, postFA) name the
# hemisphere; match their pre FA values with the whole-tract table.
rows = []
for mk, mkname in [(1, "N"), (2, "T")]:
    for tract, tname in [(1, "CB"), (3, "FM")]:
        for hemi in ["L", "R"]:
            v = dbs.sheet(f"source_data/FA_{tname}_{hemi}_mk{mkname}.xlsx", header=0)
            pre = np.round(v.iloc[:, 3].astype(float).values, 6)
            for side in (1, 2):
                w = np.round(fa[(fa.mk == mk) & (fa.tract == tract) & (fa.side == side) & (fa.prepost == 1)].FA.values, 6)
                rows.append({"monkey": mkname, "tract": tname, "sheet_hemisphere": hemi, "side": SIDE[side],
                             "sheet_voxels": len(pre), "table_voxels": len(w),
                             "same_values": len(pre) == len(w) and bool(np.allclose(np.sort(pre), np.sort(w)))})
print(pd.DataFrame(rows).to_string(index=False))

The Extended Data Fig. 4 voxel sheets (with coordinates) hold other voxel sets than the whole-tract
table for the CB (158 and 84 voxels in N against 218 and 150), so they do not show the hemisphere
of each side there. The FM sheets do: the control-side FM values of the table are the right FM of N
and the left FM of T, so side 1 is the left hemisphere in N and the right in T, the hemispheres the
paper gives for the leads. This is consistent with side 1 being the stimulated side throughout.

In [ ]:
fcNT, fcD = dbs.seed_fc("mkNT"), dbs.seed_fc("mkD")
cnt = fcNT[fcNT.time_condition == 1].groupby(["roi", "side", "subject"]).size().unstack([1, 2])
cnt.columns = [f"{SIDE[s]}_{'N' if m == 1 else 'T'}" for s, m in cnt.columns]
cnt["stim_pooled"] = cnt.stim_N + cnt.stim_T
cntD = fcD[fcD.time_condition == 1].groupby(["roi", "side"]).size().unstack()
cnt["D_side1_left"], cnt["D_side2_right"] = cntD[1], cntD[2]
cnt.index = [f"{r} {LAB[r]}" for r in cnt.index]
print("Seed connectivity voxels per target ROI (pre; post has the same counts):")
print(cnt.to_string())
print(f"\nN and T rows {len(fcNT)}, monkey D rows {len(fcD)}; sessions (subject, time, date):",
      fcNT.groupby(["subject", "time_condition", "date"]).size().to_dict(), fcD.groupby(["subject", "time_condition", "date"]).size().to_dict())
for name, d in [("N and T", fcNT), ("D", fcD)]:
    print(f"{name}: normalized connectivity {d.fc.min():.4f} to {d.fc.max():.4f}, median {d.fc.median():.4f}")
print("voxel coordinates identical between the N and T table and the D table:",
      bool(np.allclose(fcNT[["x", "y", "z"]].values[:len(fcD)], fcD[["x", "y", "z"]].values)) if len(fcNT) >= len(fcD) else None)
print("rows per voxel and session (N and T, D):",
      int(fcNT.groupby(["subject", "time_condition", "side", "roi", "x", "y", "z"]).size().max()),
      int(fcD.groupby(["time_condition", "side", "roi", "x", "y", "z"]).size().max()))
fig5_script = codes["code/Fig5BCandE9_dbs_FC_25norm_mkNT_analysisscript_a.m"].splitlines()
print("\nFig5BCandE9 script, loading the sheet:")
print("\n".join(f"{i:4d}: {l}" for i, l in enumerate(fig5_script, 1) if 7 <= i <= 15))

The voxel counts per side are the NMT-space left and right counts swapped between N and T, so
side 1 is each monkey's stimulated hemisphere, as the claims file reads it; pooled over N and T
they match the Supplementary Table 8 sums used in the claims (for example DMN below 857 per side),
and monkey D's left plus right count equals the pooled count of each side. The rows give the error
df of the paper's models (20,348 and 10,174 values). The values are all Fisher Z divided by each
session's maximum, so most lie near 0 and the largest near 1. Each voxel has one
value per session (the `date` column numbers the sessions, four for N and T and two for D), and the
script analyses the sheet as it is, with no run index. The deposited maps are therefore one per
session. The file names say "concatenated", which may refer to runs or to the subjects and ROIs
combined in one sheet, and the paper does not say whether runs were concatenated or their maps
averaged. Only this one version can be run (see deviations); its interaction F values are the
paper's. There are no seed values for the
nonsurgical controls B, W and H.

In [ ]:
conn = {k: dbs.connectomes(s) for k, s in [("DBS", "Average FC matrix DBS"), ("CTRL", "Average FC matrix CTRL"),
                                           ("D", "Sham animal FC matrix")]}
for k, c in conn.items():
    n = len(c["pre"])
    print(f"{k}: sheets {c['titles']}, {n} x {n} ROIs; symmetric {all(np.allclose(m, m.T) for m in (c['pre'], c['post'], c['diff']))}; "
          f"diagonal pre {np.unique(np.diag(c['pre']))}, diff {np.unique(np.diag(c['diff']))}; "
          f"max |post - pre - diff| {np.abs(c['post'] - c['pre'] - c['diff']).max():.1e}; "
          f"off-diagonal Z pre {np.min(c['pre'][~np.eye(n, dtype=bool)]):.3f} to {np.max(c['pre'][~np.eye(n, dtype=bool)]):.3f}, "
          f"diff {c['diff'].min():.3f} to {c['diff'].max():.3f}")
circ = dbs.sheet("source_data/Source Data FC matrix.xlsx", sheet_name="Circular plot")
print("Circular plot sheet: DBS block equals the first 54 x 54 of the DBS difference:",
      bool(np.allclose(circ.iloc[1:55, 0:54].astype(float).values, conn["DBS"]["diff"][:54, :54])),
      "; control block equals the control difference:",
      bool(np.allclose(circ.iloc[57:111, 0:54].astype(float).values, conn["CTRL"]["diff"][:54, :54])))

The connectomes are group averages (DBS: N and T; controls: B, W and H; monkey D alone) of 123
ROIs with no ROI names. The paper's per-ROI legend gives n = 122 values per ROI, which is the
number of other ROIs in a 123-ROI matrix. The circular plot uses the first 54 ROIs, which the
Fig. 4 script takes as the cortical (CHARM) block. No per-monkey connectome is deposited.

In [ ]:
cc = {r: dbs.cc1(r) for r in ["MCC_CB", "dACC_CB", "UF-FMoverlap"]}
for r, d in cc.items():
    g = d.groupby(["mk", "side"])
    print(f"{r}: fields per monkey and side {g.size().to_dict()}; sections per monkey and side {g.section.nunique().to_dict()}; "
          f"ratio {d.ratio.min():.1f} to {d.ratio.max():.1f} %; max |100 CC1/DAPI - ratio| {np.abs(100 * d.CC1 / d.DAPI - d.ratio).max():.2e}")
ccD = dbs.cc1_mkD()
mcc = cc["MCC_CB"]
bad = mcc[np.abs(100 * mcc.CC1 / mcc.DAPI - mcc.ratio) > 1e-6]
print("MCC-CB fields whose ratio is not 100 x CC1 / DAPI:")
print(bad.assign(ratio_from_counts=100 * bad.CC1 / bad.DAPI).to_string())
print("fields with more CC1 than DAPI cells:", {r: int((d.CC1 > d.DAPI).sum()) for r, d in cc.items()})
print(f"monkey D MCC-CB: {len(ccD)} rows, sections {ccD.slide.nunique()}, fields per section {ccD.groupby('slide').size().to_dict()}; "
      f"lead-side ratio {ccD.ratio_leadside.min():.1f} to {ccD.ratio_leadside.max():.1f}, control {ccD.ratio_controlside.min():.1f} to {ccD.ratio_controlside.max():.1f}")
gr = dbs.gratio()
print(f"\nEM: {len(gr)} axons; per monkey and side {gr.groupby(['mk', 'side']).size().to_dict()}")
print("image labels by monkey and side:", gr.groupby(["mk", "side"]).label.agg(lambda s: sorted(set(x[:2] for x in s))).to_dict())
print(f"aspect ratio (long / short diameter) {gr.aspect.min():.3f} to {gr.aspect.max():.3f}; "
      f"max |d_long / d_short - aspect| {np.abs(gr.d_long / gr.d_short - gr.aspect).max():.1e}")
d_mean = (gr.d_long + gr.d_short) / 2
m_mean = (gr.m_thick + gr.m_thin) / 2
print(f"column 'diameter of axon' / mean diameter: {np.median(gr.radius / d_mean):.3f}; "
      f"max |g - d / (d + 2 m)| {np.abs(gr.g - d_mean / (d_mean + 2 * m_mean)).max():.1e}; g {gr.g.min():.3f} to {gr.g.max():.3f}")

The fields match the Methods: 7 fields in each of 4 sections per hemisphere and monkey for the
MCC-CB and dACC-CB (28, 112 in all), 5 x 4 for the UF/FM overlap (20, 80 in all), and 28 for
monkey D. The ratio column is 100 x CC1 / DAPI except in two MCC-CB fields of monkey T's control
side (section T423, fields 3 and 4): the ratio of field 4 is the ratio computed from field 3's
counts, and field 3's ratio fits neither. Two MCC-CB fields have more CC1-positive than
DAPI-positive cells, which a ratio of CC1-positive cells among DAPI-positive cells cannot have.
The analyses use the ratio column as the paper's script does; the ratio from the counts is shown
as a variant. The EM sheet holds 700 axons (181 per side
in N, 169 in T; the Methods say about 175 per hemisphere and animal), all with aspect ratio below
2, so the exclusion is already applied. Its column headed "diameter of axon" holds half the mean
of the long and short diameters, and the g-ratio column equals d / (d + 2m) with d the mean
diameter and m the mean myelin width, the Methods' definition.

## ed1-behavior-increase

> After 6 weeks of SCC-DBS, monkeys N and T spent more time moving and more time foraging in the
> home cage than before, in the same direction in both animals.

ANOVA of the binary bins on condition with monkey, as in the FigE1 script (`anovan`, model
"full": monkey, condition and their interaction; reading (a)), and with monkey and condition only
(reading (b)).

In [ ]:
nt = b[b.mk < 3]
rows = []
for y in ["movement", "foraging"]:
    for reading, terms in [("a", [("mk",), ("stim",), ("mk", "stim")]), ("b", [("mk",), ("stim",)])]:
        f = dbs.anova(nt, y, ["mk", "stim"], terms)
        r = f["table"].loc["stim"]
        rows.append({"behavior": y, "reading": reading, "F": r.F, "df": f"1,{f['dfe']}", "P": r.P, "eta_p2": r.eta_p2})
print(pd.DataFrame(rows).to_string(index=False))
print("\nProbability per monkey (1 N, 2 T) and condition (0 pre, 1 post):")
print(nt.groupby(["mk", "stim"])[["movement", "foraging"]].mean().unstack().round(3).to_string())

**Result: reproduced.** Under reading (a), the paper's model, condition gives F(1,236) = 34.61
(P = 1.4e-08, partial eta squared 0.128) for movement and F(1,236) = 17.35 (P = 4.4e-05, 0.068)
for foraging, the reported values. Under reading (b) both stay significant (F 33.81 and 16.46).
The post probability is higher in both monkeys for both behaviors: movement 0.083 to 0.550 (N)
and 0.200 to 0.383 (T), foraging 0.033 to 0.050 (N) and 0.017 to 0.300 (T).

## ed1-monkey-d-no-change

> In the unstimulated lead-implanted monkey D, the amount of movement and foraging did not change
> between before and 6 weeks after implantation.

The criterion's test: a one-way ANOVA of monkey D's 120 bins on condition, at P < 0.05. A
two-sided Fisher exact test of the same counts is shown as a variant.

The FigE1 script's lines that select the rows of its ANOVA are printed first, then every line
that calls a test or selects monkey D.

In [ ]:
fige1 = codes["code/FigE1_dbs_behavior_analysisscript_a.m"].splitlines()
print("FigE1 script, the ANOVA's rows (column 6 is group, column 4 monkey):")
print("\n".join(f"{i:4d}: {l}" for i, l in enumerate(fige1, 1) if 10 <= i <= 20))
import re
print("\nEvery line of the script's", len(fige1), "that calls a test or selects monkey D (column 4 == 3, or column 6 == 2):")
print("\n".join(f"{i:4d}: {l.strip()}" for i, l in enumerate(fige1, 1)
                if re.search(r"anovan|anova1|multcompare|ranksum|signrank|ttest|fishertest|kruskalwallis|chi2", l)
                or re.search(r"\(:,4\) == 3|\(:,6\) == 2", l)))
print("groups per monkey:", b.groupby("mk").group.unique().to_dict(), "\n")
D = b[b.mk == 3]
rows = []
for y in ["movement", "foraging"]:
    f = dbs.anova(D, y, ["stim"], [("stim",)])
    r = f["table"].loc["stim"]
    tab = pd.crosstab(D.stim, D[y]).reindex(columns=[0, 1], fill_value=0).values
    rows.append({"behavior": y, "pre": D[D.stim == 0][y].mean(), "post": D[D.stim == 1][y].mean(),
                 "post_minus_pre": D[D.stim == 1][y].mean() - D[D.stim == 0][y].mean(),
                 "F": r.F, "df": f"1,{f['dfe']}", "P": r.P, "eta_p2": r.eta_p2, "fisher_P": stats.fisher_exact(tab)[1],
                 "bins_pre": int(tab[0, 1]), "bins_post": int(tab[1, 1])})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: not reproduced.** Movement does not differ (0.383 pre, 0.267 post, F(1,118) = 1.86,
P = 0.175), but foraging does: 11 of 60 bins before and 3 of 60 after (0.183 to 0.050,
F(1,118) = 5.32, P = 0.023; Fisher exact P = 0.043). The change is a decrease, the opposite of the
DBS animals. The deposited FigE1 script
(135 lines) runs its only tests, `anovan` (lines 18 and 19) and `multcompare` (24 and 26), on the
rows of group 1, monkeys N and T. Monkey D (monkey 3, group 2) enters only the means, standard
errors and sample sizes of lines 35 to 125. The deposited script therefore runs no test for
monkey D, and the paper reports none; the test here is the criterion's. Lines 93 and 96 compute
monkey D's foraging standard errors from the movement column (column 2).


## fig2a-cb-fa-increase

> After 6 weeks of SCC-DBS, fractional anisotropy (FA) increased in the cingulum bundle of the
> stimulated hemisphere, with a significant PrePost x side interaction and a significant pre
> against post difference in the stimulated CB.

Model of the Fig2A script: monkey, PrePost, side, tract, and all interactions of PrePost, side and
tract; type III sums of squares (`anovan` default), type II as a variant. Tukey-Kramer over the 12
PrePost x side x tract cells (`multcompare`, Dimension [2 3 4]).

In [ ]:
TRACT = {1: "CB", 2: "UF", 3: "FM"}
fit_fa = {ss: dbs.anova(fa, "FA", ["mk", "prepost", "side", "tract"], T("mk", "prepost", "side", "tract"), sstype=ss) for ss in (3, 2)}
for ss, f in fit_fa.items():
    print(f"type {ss} sums of squares, error df {f['dfe']}:")
    print(f["table"].round(5).to_string())
pp_fa = dbs.prepost_pairs(dbs.tukey_kramer(fit_fa[3], ["prepost", "side", "tract"], only_vary="prepost"), "tract")
pp_fa["side"], pp_fa["tract"] = pp_fa.side.map(SIDE), pp_fa.tract.map(TRACT)
print("\nTukey-Kramer, pre against post (population marginal means):")
print(pp_fa.to_string(index=False))

**Result: reproduced.** The PrePost x side interaction is F(1,3187) = 4.64, P = 0.031, partial
eta squared 0.0015, the reported values (type II: F = 6.34, P = 0.012). The stimulated CB rises
from 0.324 to 0.369 (Tukey-Kramer P = 0.022, the reported P).

## fig2a-no-change-other-tracts

> FA did not change in the stimulated UF or FM, nor in any of the three tracts of the control
> hemisphere.

In [ ]:
others = pp_fa[~((pp_fa.side == "stim") & (pp_fa.tract == "CB"))]
print(others.to_string(index=False))
print("comparisons with P < 0.05:", int((others.P < 0.05).sum()), "of", len(others))

**Result: reproduced.** None of the five other pre against post comparisons has P < 0.05; the
smallest is control FM (change -0.037, P = 0.86), and the stimulated UF and FM have P = 1.00.

## ed2-cb-both-monkeys

> The FA increase in the stimulated CB was in the same direction in both monkeys.

In [ ]:
cb = fa[(fa.tract == 1)]
per = cb.groupby(["side", "mk", "prepost"]).FA.mean().unstack()
per["post_minus_pre"] = per[2] - per[1]
per["voxels"] = cb[cb.prepost == 1].groupby(["side", "mk"]).size()
per.index = [f"{SIDE[s]} {'N' if m == 1 else 'T'}" for s, m in per.index]
print(per.rename(columns={1: "pre", 2: "post"}).round(4).to_string())

**Result: reproduced.** Mean stimulated-CB FA rises by 0.019 in N (218 voxels) and 0.079 in T
(162 voxels).

## fig2b-mcc-cb-fa-increase

> Of the four CB subregions, only the mid-cingulate portion (MCC-CB) of the stimulated hemisphere
> showed an FA increase, and it did so in both animals.

The subregion labels are the dataset's (column `part`), and the voxel counts match the legend
(data check). Model of the Fig2B script, Tukey-Kramer over the 16 cells.

In [ ]:
PART = {1: "SCC", 2: "dACC", 3: "MCC", 4: "PCC"}
fit_sp = {ss: dbs.anova(sp, "FA", ["mk", "prepost", "side", "part"], T("mk", "prepost", "side", "part"), sstype=ss) for ss in (3, 2)}
for ss, f in fit_sp.items():
    print(f"type {ss}, error df {f['dfe']}:", f["table"].loc["prepost x side"].round(5).to_dict())
pp_sp = dbs.prepost_pairs(dbs.tukey_kramer(fit_sp[3], ["prepost", "side", "part"], only_vary="prepost"), "part")
pp_sp["side"], pp_sp["part"] = pp_sp.side.map(SIDE), pp_sp.part.map(PART)
print(pp_sp.to_string(index=False))
per = sp.groupby(["side", "part", "mk", "prepost"]).FA.mean().unstack()
per = (per[2] - per[1]).unstack("mk").rename(columns={1: "N post-pre", 2: "T post-pre"})
per.index = [f"{SIDE[s]} {PART[p]}" for s, p in per.index]
print("\nMean post minus pre FA per monkey:")
print(per.round(4).to_string())

**Result: reproduced.** PrePost x side F(1,1357) = 5.42, P = 0.020, partial eta squared 0.0040,
the reported values (type II: F = 4.03, P = 0.045). The stimulated MCC-CB rises from 0.416 to
0.551 (Tukey-Kramer P = 0.0018, the reported 0.0017); no other subregion and side has P < 0.05
(smallest 0.91). The MCC-CB change is positive in N (0.081) and T (0.205).

## fig3c-cc1-dapi-ratio

> The proportion of CC1-positive (myelinating oligodendrocyte) cells among DAPI-positive cells in
> the MCC-CB was higher in the stimulated than in the control hemisphere, in both animals.

Fig3CandE5 script: `anovan` model "full" on monkey and side (reading (a)); reading (b) without the
interaction.

In [ ]:
def side_anova(d, y):
    rows = []
    for reading, terms in [("a", [("mk",), ("side",), ("mk", "side")]), ("b", [("mk",), ("side",)])]:
        f = dbs.anova(d, y, ["mk", "side"], terms)
        r = f["table"].loc["side"]
        rows.append({"measure": y, "reading": reading, "F": r.F, "df": f"1,{f['dfe']}", "P": r.P, "eta_p2": r.eta_p2})
    return rows

def per_monkey(d, y):
    m = d.groupby(["mk", "side"])[y].mean().unstack().rename(columns={1: "stim", 2: "control"})
    m["stim_minus_control"] = m.stim - m.control
    m.index = ["N" if i == 1 else "T" for i in m.index]
    return m

print(pd.DataFrame(side_anova(cc["MCC_CB"], "ratio")).to_string(index=False))
print(per_monkey(cc["MCC_CB"], "ratio").round(3).to_string())
recomputed = cc["MCC_CB"].assign(ratio_from_counts=100 * cc["MCC_CB"].CC1 / cc["MCC_CB"].DAPI)
print("\nvariant, ratio computed from the counts:")
print(pd.DataFrame(side_anova(recomputed, "ratio_from_counts")).to_string(index=False))

**Result: reproduced.** Side gives F(1,108) = 9.00, P = 0.0034, partial eta squared 0.077 (the
reported values) under reading (a) and F(1,109) = 9.08, P = 0.0032 under (b). The stimulated
side is higher in N (80.6 against 75.0 %) and T (55.1 against 50.1 %). With the ratio computed
from the counts, F = 9.03, P = 0.0033.

## ed5-cc1-count-dapi-unchanged

> In the MCC-CB the number of CC1-positive cells was higher on the stimulated side, while the
> number of DAPI-positive cells did not differ between hemispheres.

In [ ]:
print(pd.DataFrame(side_anova(cc["MCC_CB"], "CC1") + side_anova(cc["MCC_CB"], "DAPI")).to_string(index=False))
print("CC1:"); print(per_monkey(cc["MCC_CB"], "CC1").round(2).to_string())
print("DAPI:"); print(per_monkey(cc["MCC_CB"], "DAPI").round(2).to_string())

**Result: reproduced.** The CC1 count has a side effect, F(1,108) = 6.22, P = 0.014, partial
eta squared 0.054 (reading (b): F = 6.17, P = 0.014), higher on the stimulated side in N (306.1
against 281.0) and T (186.5 against 178.8). The DAPI count has none (F = 0.55, P = 0.46; (b)
P = 0.47).

## ed6-dacc-ufm-no-oligodendrocyte-change

> Neither the CC1/DAPI ratio nor the numbers of CC1-positive and DAPI-positive cells differed
> between hemispheres in the anterior (dACC) portion of the CB, nor in the region where the FM and
> UF overlap.

In [ ]:
rows = []
for r in ["dACC_CB", "UF-FMoverlap"]:
    for y in ["ratio", "CC1", "DAPI"]:
        for row in side_anova(cc[r], y):
            m = per_monkey(cc[r], y)
            rows.append({"region": r, **row, "N_stim_minus_control": m.stim_minus_control["N"],
                         "T_stim_minus_control": m.stim_minus_control["T"]})
res = pd.DataFrame(rows)
print(res.round(4).to_string(index=False))
print("reading (a) comparisons with P < 0.05:", int((res[res.reading == "a"].P < 0.05).sum()), "of 6")

**Result: reproduced.** None of the six reading (a) tests has P < 0.05: dACC-CB ratio P = 0.090,
CC1 P = 0.997, DAPI P = 0.27; UF/FM overlap ratio P = 0.23, CC1 P = 0.38, DAPI P = 0.95.
Reading (b) gives the same P values to two digits. The F values match the Extended Data Fig. 6
legend; the UF/FM DAPI P is 0.9511 here against the legend's 0.9611, with F = 0 in both.

## fig3f-g-ratio-decrease

> The g-ratio of myelinated axons in the MCC-CB was lower in the stimulated than in the control
> hemisphere, in both animals, meaning thicker myelin.

Fig3F script: `anovan` model "full" on monkey and side, one g-ratio per axon (the sheet's g-ratio
column, checked against the Methods' formula in the data check).

In [ ]:
print(pd.DataFrame(side_anova(gr, "g")).to_string(index=False))
print(per_monkey(gr, "g").round(4).to_string())
print("axons:", gr.groupby(["mk", "side"]).size().to_dict())

**Result: reproduced.** Side gives F(1,696) = 58.98, P = 5.4e-14, partial eta squared 0.078
(the reported values); reading (b) F(1,697) = 59.83. The g-ratio is lower on the stimulated
side in N (0.600 against 0.684) and T (0.618 against 0.642).

## fig4a-cortical-down-subcortical-up

> After SCC-DBS, whole-brain functional connectivity between cortical areas decreased and
> connectivity between subcortical areas increased.

The deposited DBS group difference connectome (post minus pre, mean of N and T). Cortical ROIs are
the first 54 (Fig. 4 script), subcortical the other 69. Reading (a): mean of the off-diagonal
entries within each block; reading (b): each ROI's mean change to all other ROIs, median over the
ROIs of each block.

In [ ]:
def blocks(m, nc=dbs.N_CORTICAL):
    n = len(m)
    cort = m[:nc, :nc][np.triu_indices(nc, 1)]
    sub = m[nc:, nc:][np.triu_indices(n - nc, 1)]
    roi = np.array([dbs.offdiag(m, i).mean() for i in range(n)])
    return {"a_cortical_block_mean": cort.mean(), "a_subcortical_block_mean": sub.mean(),
            "b_cortical_roi_median": np.median(roi[:nc]), "b_subcortical_roi_median": np.median(roi[nc:]),
            "cortical_entries_negative": (cort < 0).mean(), "subcortical_entries_negative": (sub < 0).mean(),
            "cortico_subcortical_mean": m[:nc, nc:].mean(), "diff_min": m.min(), "diff_max": m.max()}

block_table = pd.DataFrame({k: blocks(c["diff"]) for k, c in conn.items()})
print(block_table.round(4).to_string())

**Result: reproduced.** In the DBS group difference connectome the cortical block mean is
-0.096 and the subcortical block mean +0.012 (reading (a)); the median ROI change is -0.052 over
cortical ROIs and +0.006 over subcortical ROIs (reading (b)). 78% of cortical entries are
negative, 40% of subcortical entries.

## fig4b-roi-changes-counts

> Per ROI, functional connectivity changed significantly between pre and post scans in 69 of the
> 123 CHARM and SARM ROIs in the DBS group and in 44 of 123 in the nonsurgical control group.

The criterion, amended after the reader's answer of 2026-11-03, is the test of the paper's Fig. 4
script: for each of the 123 ROIs, a two-sided rank-sum of its 122 off-diagonal changes against 122
zeros, `ranksum(sbj_data_mean, zeros(length(sbj_data_mean),1))`, counted when P < 0.05/123 (the
script's 0.05/nrois). The rank-sum is computed as MATLAB `ranksum` does for these sample sizes
(normal approximation with tie correction and continuity correction, `scipy.stats.mannwhitneyu`,
method "asymptotic"). The tolerance is 62.1 to 75.9 (DBS) and 39.6 to 48.4 (controls).

Reported as variants, not judged on: the Methods' threshold P < 0.01/123, a rank-sum of the ROI's
pre values against its post values, and a signed-rank test of its changes against zero; also the
thresholds 0.01/122 and 0.01 uncorrected, and the values with the ROI's own diagonal, which the
earlier criterion named.

In [ ]:
def mw(a, b):
    return stats.mannwhitneyu(a, b, alternative="two-sided", method="asymptotic", use_continuity=True).pvalue

def roi_pvalues(c, keep_diag):
    n = len(c["pre"])
    out = {}
    out["a rank-sum pre vs post"] = np.array([mw(dbs.offdiag(c["pre"], i, keep_diag), dbs.offdiag(c["post"], i, keep_diag)) for i in range(n)])
    out["b signed-rank change vs 0"] = np.array([stats.wilcoxon(dbs.offdiag(c["diff"], i, keep_diag), zero_method="wilcox").pvalue for i in range(n)])
    out["code rank-sum change vs zeros"] = np.array([mw(dbs.offdiag(c["diff"], i, keep_diag), np.zeros(n - 1 + keep_diag)) for i in range(n)])
    return out

pvals = {}
rows = []
for grp in ["DBS", "CTRL"]:
    n = len(conn[grp]["pre"])
    for keep_diag in (False, True):
        pv = roi_pvalues(conn[grp], keep_diag)
        pvals[(grp, keep_diag)] = pv
        for test, p in pv.items():
            for thr_name, thr in [("0.01/122", 0.01 / 122), ("0.01 uncorrected", 0.01), ("0.01/123", 0.01 / n), ("0.05/123 (code)", 0.05 / n)]:
                rows.append({"group": grp, "values": f"{n - 1 + keep_diag}" + (" with diagonal" if keep_diag else " off-diagonal"),
                             "test": test, "threshold": thr_name, "significant": int((p < thr).sum())})
counts = pd.DataFrame(rows).pivot_table(index=["values", "test", "threshold"], columns="group", values="significant").astype(int)
counts["DBS_in_tol"] = counts.DBS.between(62.1, 75.9)
counts["CTRL_in_tol"] = counts.CTRL.between(39.6, 48.4)
print(counts.to_string())

In [ ]:
p_code = pvals[("DBS", False)]["code rank-sum change vs zeros"]
pc = pvals[("CTRL", False)]["code rank-sum change vs zeros"]
n_dbs, n_ctrl = int((p_code < 0.05 / 123).sum()), int((pc < 0.05 / 123).sum())
print(f"criterion (rank-sum of the 122 off-diagonal changes against zeros, P < 0.05/123): DBS {n_dbs} (tolerance 62.1 to 75.9), "
      f"controls {n_ctrl} (39.6 to 48.4); both in tolerance: {62.1 <= n_dbs <= 75.9 and 39.6 <= n_ctrl <= 48.4}")
sig = p_code < 0.05 / 123
print(f"DBS: largest P among the significant ROIs {p_code[sig].max():.2e}; smallest P among the others {p_code[~sig].min():.2e}")
print(f"DBS significant ROIs, cortical {int(sig[:54].sum())} of 54, subcortical {int(sig[54:].sum())} of 69; "
      f"of them with negative mean change {int((sig & (np.array([dbs.offdiag(conn['DBS']['diff'], i).mean() for i in range(123)]) < 0)).sum())}")
print(f"controls: largest significant P {pc[pc < 0.05 / 123].max():.2e}; smallest P among the others {pc[pc >= 0.05 / 123].min():.2e}")
print(f"thresholds: 0.05/123 = {0.05 / 123:.2e}, 0.01/123 = {0.01 / 123:.2e}")

**Result: reproduced.** Under the amended criterion, the rank-sum of each ROI's 122 changes against
zeros at P < 0.05/123 gives 69 ROIs in the DBS group and 44 in the controls, the reported counts
(with the diagonal, 69 and 44 as well). The largest P counted as significant is 3.88e-04 in both
groups, the largest value in Supplementary Tables 1 and 2 (3.9e-4). Of the variants, the same test
at the Methods' threshold 0.01/123 gives 62 and 36, the pre against post rank-sum 10 to 44 (DBS)
and 4 to 15 (controls), and the signed-rank test 60 to 85 and 27 to 49.

## fig4c-network-sorted-reduction

> Sorted by cortical network membership, functional connectivity within and across networks was
> reduced after SCC-DBS.

The criterion needs the paper's code to assign the connectome's ROIs to networks. The Fig. 4
script's lines that name ROIs or draw the circular plot, and every text cell of the connectome
workbook, are printed.

In [ ]:
fig4_script = codes["code/Fig4 Fujimoto_NatNeuro_2026_FCmatrix.m"].splitlines()
for i, line in enumerate(fig4_script, 1):
    if any(k in line.lower() for k in ("key", "reorder", "label_names", "circulargraph", "mycolormap", "network")):
        print(f"{i:4d}: {line.strip()}")
book = dbs.sheet("source_data/Source Data FC matrix.xlsx", sheet_name=None)
for name, d in book.items():
    print(f"sheet {name!r}, {d.shape}: text cells", [(r, c, v) for (r, c), v in np.ndenumerate(d.values) if isinstance(v, str)])

**Result: blocked (methods_underspecified).** The script reads the ROI names from atlas key
files and an order file (CHARM_key_4.txt, SARM_key_4.txt, reorder_rois.txt) that are not
deposited, draws the circular plot of the first 54 ROIs with one colour for decreases and one for
increases, and assigns no ROI to a network. The workbook's only text cells are block titles.
Supplementary Tables 3 and 4 name the ROIs but give no network. Without an assignment the
criterion blocks the claim.

## fig4-nonsurgical-no-pattern

> In the nonsurgical control animals, scans 6 to 7 weeks apart showed no systematic pattern of
> functional connectivity change.

The measure of fig4a-cortical-down-subcortical-up on the control group's difference connectome
(column CTRL of the table above).

In [ ]:
print(block_table[["DBS", "CTRL"]].round(4).to_string())
ctrl = block_table.CTRL
for r in ["a", "b"]:
    k = ["a_cortical_block_mean", "a_subcortical_block_mean"] if r == "a" else ["b_cortical_roi_median", "b_subcortical_roi_median"]
    print(f"reading ({r}): DBS pattern (cortical < 0 and subcortical > 0) present in the controls: {bool(ctrl[k[0]] < 0 and ctrl[k[1]] > 0)}")
print("significant ROIs in the controls under the code's reading:", int((pvals[("CTRL", False)]["code rank-sum change vs zeros"] < 0.05 / 123).sum()))

**Result: reproduced.** In the controls the cortical block mean is +0.001 and the subcortical
-0.004 (reading (a)), the ROI medians +0.001 and -0.001 (reading (b)); the DBS pattern (cortical
negative, subcortical positive) is absent under both. 49% of the controls' cortical entries are
negative, against 78% in the DBS group. The code's per-ROI test counts 44 significant ROIs in
the controls.

## fig5a-pre-scc-connectivity

> Before stimulation, the SCC was positively connected with vmFC, dACC, PCC, STG, hippocampus,
> amygdala and DLPFC, and negatively with the insula and somatosensory cortex.

Pre-DBS normalized seed connectivity, mean over the ROI's voxels pooled over N and T: (a) in the
stimulated hemisphere; (b) both hemispheres pooled. Only one seed map per session is deposited
(see the data check), so the run readings cannot both be run.

In [ ]:
want = {"vmFC": 1, "dACC": 1, "PCC": 1, "STG": 1, "HIP": 1, "AMY": 1, "DLPFC": 1, "Anterior-dorsal Insula": -1, "Sensory Cx": -1}
num = {v: k for k, v in LAB.items()}
pre = fcNT[fcNT.time_condition == 1]
rows = []
for name, sign in want.items():
    r = num[name]
    a = pre[(pre.roi == r) & (pre.side == 1)].fc
    bb = pre[pre.roi == r].fc
    rows.append({"roi": name, "expected_sign": sign, "a_stim_side_mean": a.mean(), "b_both_sides_mean": bb.mean(),
                 "N_stim": pre[(pre.roi == r) & (pre.side == 1) & (pre.subject == 1)].fc.mean(),
                 "T_stim": pre[(pre.roi == r) & (pre.side == 1) & (pre.subject == 2)].fc.mean(),
                 "a_match": np.sign(a.mean()) == sign, "b_match": np.sign(bb.mean()) == sign})
res = pd.DataFrame(rows)
print(res.round(4).to_string(index=False))
print("signs matching: reading (a)", int(res.a_match.sum()), "of 9; reading (b)", int(res.b_match.sum()), "of 9")

**Result: reproduced.** Before DBS the seven ROIs are positive (vmFC 0.123, STG 0.099, DLPFC
0.066, dACC 0.064, AMY 0.042, HIP 0.038, PCC 0.038 in the stimulated hemisphere) and the
anterior-dorsal insula (-0.013) and somatosensory cortex (-0.026) negative; 9 of 9 signs match
in the stimulated hemisphere and 9 of 9 with both hemispheres pooled.

## fig5b-scc-decreases

> After SCC-DBS, functional connectivity of the stimulated SCC decreased with vmFC, PCC, STG,
> hippocampus, amygdala, DLPFC and dACC in the stimulated hemisphere, with a significant PrePost x
> side interaction.

Model of the Fig5BCandE9 script on the 17 target ROIs: subject, PrePost, side, ROI, and all
interactions of PrePost, side and ROI. Tukey-Kramer over the 68 cells. SciPy's studentized range
tail loses accuracy below about 1e-7 (the table before the summary compares it with the reported
P values) and does not resolve P values below about 1e-11 (MATLAB prints them as 0); such P values are shown at
that floor.

In [ ]:
fit_roi = {ss: dbs.anova(fcNT, "fc", ["subject", "time_condition", "side", "roi"], T("subject", "time_condition", "side", "roi"), sstype=ss) for ss in (3, 2)}
for ss, f in fit_roi.items():
    print(f"type {ss}, error df {f['dfe']}:")
    print(f["table"].to_string())
pp_roi = dbs.prepost_pairs(dbs.tukey_kramer(fit_roi[3], ["time_condition", "side", "roi"], only_vary="time_condition"), "roi", "time_condition")
pp_roi["name"] = pp_roi.roi.map(LAB)
pp_roi["side"] = pp_roi.side.map(SIDE)
print("\nTukey-Kramer pre against post, 68 cells:")
print(pp_roi.to_string(index=False))
seven = ["vmFC", "PCC", "STG", "HIP", "AMY", "DLPFC", "dACC"]
s7 = pp_roi[(pp_roi.side == "stim") & pp_roi.name.isin(seven)]
print("\nstimulated side, the seven ROIs: decreases with P < 0.05:", int(((s7.P < 0.05) & (s7.post_minus_pre < 0)).sum()), "of 7")

**Result: reproduced.** PrePost x side F(1,20279) = 186.48, P = 2.9e-42, partial eta squared
0.0091, the reported values (type II: F = 56.2, P = 6.7e-14). All seven stimulated-side ROIs
decrease with P < 0.001: dACC P = 4.9e-04 (reported 4.75e-04), HIP 1.3e-05 (1.19e-05), AMY
8.8e-06 (7.70e-06), PCC 4.5e-11 (8.22e-12), and vmFC, STG and DLPFC at the floor of the computation
(1.4e-11; printed as 0 in the paper).

## fig5b-scc-increases

> After SCC-DBS, functional connectivity of the stimulated SCC increased with the anterior-dorsal
> insula and the somatosensory cortex.

In [ ]:
print(pp_roi[(pp_roi.side == "stim") & pp_roi.name.isin(["Anterior-dorsal Insula", "Sensory Cx"])].to_string(index=False))

**Result: reproduced.** The stimulated-side anterior-dorsal insula rises from -0.013 to +0.026
(P = 6.8e-08, reported 4.42e-08) and the somatosensory cortex from -0.026 to +0.006 (P at the
floor, 1.4e-11; printed as 0).

## ed9-control-hemisphere-changes

> Although stimulation was unilateral, SCC functional connectivity also changed in the
> unstimulated control hemisphere, including vmFC, PCC, anterior-dorsal insula and dACC.

In [ ]:
ctl = pp_roi[pp_roi.side == "control"]
print(ctl.to_string(index=False))
print("\ncontrol-side ROIs with P < 0.05:", int((ctl.P < 0.05).sum()), "of 17:", ctl[ctl.P < 0.05].name.tolist())
print("of vmFC, PCC, insula and dACC:", int((ctl[ctl.name.isin(["vmFC", "PCC", "Anterior-dorsal Insula", "dACC"])].P < 0.05).sum()), "of 4")

**Result: reproduced.** On the control side vmFC, PCC, dACC (decreases) and the anterior-dorsal
insula (increase, P = 0.029, reported 0.0293) all have P < 0.05, and 11 of 17 control-side ROIs
do, the same 11 ROIs as Supplementary Table 9.

## ed9-no-change-motor-mcc

> Connectivity between the stimulated SCC and motor-related regions (primary motor cortex, SMA)
> and the MCC did not change.

In [ ]:
print(pp_roi[(pp_roi.side == "stim") & pp_roi.name.isin(["Motor Cx", "SMA", "MCC"])].to_string(index=False))

**Result: reproduced.** Stimulated-side motor cortex P = 0.99992, SMA P = 1.00000 and MCC
P = 0.99180 (reported 0.99991, 0.99999 and 0.99183); changes +0.006, -0.002 and -0.014.

## ed7-decreases-both-monkeys

> The reductions in SCC functional connectivity after DBS were consistent across both monkeys.

In [ ]:
s = fcNT[(fcNT.side == 1) & fcNT.roi.isin([num[n] for n in seven])]
per = s.groupby(["roi", "subject", "time_condition"]).fc.mean().unstack()
per = (per[2] - per[1]).unstack("subject").rename(columns={1: "N post-pre", 2: "T post-pre"})
per.index = [LAB[r] for r in per.index]
print(per.round(4).to_string())
print("negative: N", int((per["N post-pre"] < 0).sum()), "of 7; T", int((per["T post-pre"] < 0).sum()), "of 7")

**Result: reproduced.** All seven stimulated-side ROIs decrease in N and in T (7 of 7 each); the
smallest decrease is AMY in N (-0.0006).

## fig5c-dmn-lim-cen-decrease

> After SCC-DBS, functional connectivity of the stimulated SCC with the hubs of the default mode,
> limbic and central executive networks decreased, with a significant PrePost x side interaction.

Network voxels are the hub ROIs' voxels with repeats (Fig5BCandE9 script). LIM under (a) with area
25 (the script's list and the Fig. 5c n) and (b) without. Tukey-Kramer over the 24 cells.

In [ ]:
NET = {i: k for i, k in enumerate(dbs.NETWORKS, start=1)}
pp_net, fit_net = {}, {}
for reading, lim25 in [("a", True), ("b", False)]:
    nd = dbs.network_data(fcNT, lim_with_area25=lim25)
    for ss in (3, 2):
        f = dbs.anova(nd, "fc", ["subject", "time_condition", "side", "network"], T("subject", "time_condition", "side", "network"), sstype=ss)
        fit_net[(reading, ss)] = f
        print(f"LIM reading ({reading}), type {ss}, error df {f['dfe']}:", f["table"].loc["time_condition x side"].to_dict())
    p = dbs.prepost_pairs(dbs.tukey_kramer(fit_net[(reading, 3)], ["time_condition", "side", "network"], only_vary="time_condition"), "network", "time_condition")
    p["side"], p["network"] = p.side.map(SIDE), p.network.map(NET)
    p["voxels_per_side_pre"] = [int(((nd.network == list(NET.values()).index(n) + 1) & (nd.side == (1 if s == "stim" else 2)) & (nd.time_condition == 1)).sum()) for s, n in zip(p.side, p.network)]
    pp_net[reading] = p
    print(p.to_string(index=False))

**Result: reproduced.** With LIM as the script and the Fig. 5c n have it (reading (a), 821
voxels), PrePost x side F(1,21967) = 51.09, P = 9.1e-13, partial eta squared 0.0023, the
reported values (type II: F = 40.5). Without area 25 (reading (b), 803 voxels) F(1,21895) =
61.60. Under both, the stimulated-side DMN, LIM and CEN decrease (by 0.064, 0.025 to 0.029, and
0.035) at the floor of the computation (P below 1.7e-11).

## fig5c-smn-increase

> SCC connectivity with the sensorimotor network increased after SCC-DBS, although that network
> has few direct CB connections.

In [ ]:
for r, p in pp_net.items():
    print(f"LIM reading ({r}):"); print(p[p.network == "SMN"].to_string(index=False))

**Result: reproduced.** The stimulated-side SMN rises from -0.017 to +0.003 with P below 1.7e-11
under both LIM readings (the control side too, +0.017).

## fig5c-vis-no-change

> SCC connectivity with the visual network, and with the areas within it, did not change after
> SCC-DBS.

In [ ]:
for r, p in pp_net.items():
    print(f"LIM reading ({r}), VIS:"); print(p[p.network == "VIS"].to_string(index=False))
print("\nareas (17-ROI model):")
print(pp_roi[pp_roi.name.isin(["Visual Cx", "MT"])].to_string(index=False))

**Result: reproduced.** VIS does not change on either side (stimulated P = 1.00; control
P = 0.65 with LIM reading (a), 0.23 with (b)), nor do the stimulated-side visual cortex
(P = 1.00) and MT (P = 0.999). The control-side MT decreases (P = 1.0e-06, reported 7.85e-7).

## fig5c-san-rebalanced

> In the stimulated hemisphere, the salience network as a whole showed no change in connectivity
> with the stimulated SCC, because its anterior-dorsal insula node increased and its dACC node
> decreased.

In [ ]:
for r, p in pp_net.items():
    print(f"LIM reading ({r}), SAN:"); print(p[p.network == "SAN"].to_string(index=False))
print("\nnodes, stimulated side (17-ROI model):")
print(pp_roi[(pp_roi.side == "stim") & pp_roi.name.isin(["Anterior-dorsal Insula", "dACC"])].to_string(index=False))

**Result: reproduced.** The stimulated-side SAN does not change (+0.0002, P = 1.00 under both
LIM readings), while its anterior-dorsal insula node increases (P = 6.8e-08) and its dACC node
decreases (P = 4.9e-04). The control-side SAN decreases (P = 6.9e-07, reported 6.20e-07).

## fig6a-monkey-d-cb-fa-decrease

> In the unstimulated lead-implanted monkey D, FA decreased in the CB of the implanted hemisphere
> and did not change in the other tracts.

Fig6A script: the model of fig2a-cb-fa-increase without the monkey term. Side 1 is the lead
(sham) side, the left hemisphere, as the sheet's legend says.

In [ ]:
fit_faD = {ss: dbs.anova(faD, "FA", ["prepost", "side", "tract"], T(None, "prepost", "side", "tract"), sstype=ss) for ss in (3, 2)}
for ss, f in fit_faD.items():
    print(f"type {ss}, error df {f['dfe']}:"); print(f["table"].round(5).to_string())
pp_faD = dbs.prepost_pairs(dbs.tukey_kramer(fit_faD[3], ["prepost", "side", "tract"], only_vary="prepost"), "tract")
pp_faD["side"], pp_faD["tract"] = pp_faD.side.map({1: "sham", 2: "control"}), pp_faD.tract.map(TRACT)
print(pp_faD.to_string(index=False))

**Result: reproduced.** The PrePost x side x tract interaction is F(2,1238) = 6.02, P = 0.0025,
partial eta squared 0.0096; the F, P and effect size are the reported ones, with 2 numerator df
(the legend prints 1). The sham-side CB falls from 0.362 to 0.245 (P = 8.4e-09, reported
5.07e-09); none of the five other comparisons has P < 0.05 (smallest control FM, P = 0.088).

## fig6b-monkey-d-dacc-cb-decrease

> In monkey D the CB FA decrease was significant in the dACC portion of the implanted hemisphere,
> more anterior than and opposite in direction to the MCC-CB increase after stimulation, although
> reductions were diffuse across the CB.

In [ ]:
fit_spD = {ss: dbs.anova(spD, "FA", ["prepost", "side", "part"], T(None, "prepost", "side", "part"), sstype=ss) for ss in (3, 2)}
for ss, f in fit_spD.items():
    print(f"type {ss}, error df {f['dfe']}:", f["table"].loc["prepost x side"].round(6).to_dict())
pp_spD = dbs.prepost_pairs(dbs.tukey_kramer(fit_spD[3], ["prepost", "side", "part"], only_vary="prepost"), "part")
pp_spD["side"], pp_spD["part"] = pp_spD.side.map({1: "sham", 2: "control"}), pp_spD.part.map(PART)
print(pp_spD.to_string(index=False))

**Result: reproduced.** PrePost x side F(1,492) = 24.55, P = 1.0e-06, partial eta squared 0.048,
the reported values. The sham-side dACC-CB falls from 0.413 to 0.230 (P = 3.6e-08, reported
1.30e-08); the sham-side MCC-CB falls too (-0.115, P = 0.44), so it shows no increase. All four
sham-side subregions decrease (by 0.080 to 0.184), the diffuse reduction the statement
describes; only the dACC-CB reaches P < 0.05.

## fig6c-monkey-d-cc1-no-change

> In monkey D, the CC1/DAPI ratio in the MCC-CB did not differ between the implanted and control
> hemispheres.

Fig6C script: `ttest` of the lead-side against the control-side column, paired by row. Each row
carries one section and field number (reading (a)).

In [ ]:
t = stats.ttest_rel(ccD.ratio_leadside, ccD.ratio_controlside)
dlt = ccD.ratio_leadside - ccD.ratio_controlside
print(f"paired t({t.df}) = {t.statistic:.4f}, P = {t.pvalue:.4f}; mean difference {dlt.mean():.3f} %, Cohen's dz {dlt.mean() / dlt.std(ddof=1):.3f}")
u = stats.ttest_ind(ccD.ratio_leadside, ccD.ratio_controlside)
print(f"unpaired variant: t = {u.statistic:.4f}, P = {u.pvalue:.4f}")
print(f"means: lead side {ccD.ratio_leadside.mean():.2f} %, control {ccD.ratio_controlside.mean():.2f} %; pairs {len(ccD)}")
print("columns:", list(ccD.columns))
print("fields per section (each row is one section and field, both hemispheres):",
      ccD.groupby("slide").location.apply(list).to_dict(), "; distinct section and field pairs:",
      len(ccD[["slide", "location"]].drop_duplicates()))
fig6c = codes["code/Fig6C_dbs_IF_CC1_mkD_MCC_CB_analysisscript_a.m"].splitlines()
print("Fig6C script:"); print("\n".join(f"{i:4d}: {l}" for i, l in enumerate(fig6c, 1) if 9 <= i <= 15))

**Result: reproduced.** Paired t(27) = -0.9888, P = 0.332, Cohen's dz -0.19, the reported values;
the lead side is 58.0 % and the control side 59.9 %. The unpaired variant gives P = 0.47.

## ed10c-monkey-d-subcortical-decrease

> In monkey D, whole-brain connectivity changes after implantation were less coherent than after
> SCC-DBS, cortico-cortical changes were mixed, and subcortico-subcortical connectivity decreased,
> the opposite of the SCC-DBS group.

In [ ]:
print(block_table[["DBS", "D"]].round(4).to_string())

**Result: reproduced.** In monkey D's difference connectome the subcortical block mean is
-0.035 (reading (a)) and the median subcortical ROI change -0.022 (reading (b)), with 76% of
subcortical entries negative, against +0.012 and 40% in the DBS group. Cortical changes are
mixed: block mean -0.014, 51% of entries negative (DBS group 78%).

## fig6e-monkey-d-no-network-reduction

> In monkey D the reductions in SCC connectivity with the DMN, LIM and CEN seen after SCC-DBS were
> not observed.

The model of fig5c-dmn-lim-cen-decrease on monkey D without the subject term, LIM readings (a) and
(b); "not observed" read as (a) the sham side and (b) both sides.

In [ ]:
pp_netD = {}
for reading, lim25 in [("a", True), ("b", False)]:
    nd = dbs.network_data(fcD, lim_with_area25=lim25)
    f = dbs.anova(nd, "fc", ["time_condition", "side", "network"], T(None, "time_condition", "side", "network"))
    print(f"LIM reading ({reading}), error df {f['dfe']}:", f["table"].loc["time_condition x side"].to_dict())
    p = dbs.prepost_pairs(dbs.tukey_kramer(f, ["time_condition", "side", "network"], only_vary="time_condition"), "network", "time_condition")
    p["side"], p["network"] = p.side.map({1: "sham", 2: "control"}), p.network.map(NET)
    pp_netD[reading] = p
    print(p.to_string(index=False))
    for scope, sides in [("sham side", ["sham"]), ("both sides", ["sham", "control"])]:
        q = p[p.network.isin(["DMN", "LIM", "CEN"]) & p.side.isin(sides)]
        print(f"  {scope}: significant decreases in DMN, LIM, CEN: {int(((q.P < 0.05) & (q.post_minus_pre < 0)).sum())} of {len(q)}")
lim = dbs.NETWORKS["LIM"]
for side, n_legend in [(1, 438), (2, 383)]:
    v = fcD[(fcD.time_condition == 1) & (fcD.side == side)].groupby("roi").size()
    print(f"side {side}: LIM voxels with area 25 {v[lim].sum()}, without {v[[r for r in lim if r != dbs.AREA25]].sum()}; Fig. 6e legend {n_legend}")

**Result: blocked, the readings disagree.** With LIM as the script has it (with area 25), no
sham-side DMN, LIM or CEN comparison is a significant decrease (0 of 3), but on both sides one
is: the control-side CEN (-0.009, P = 0.0089, the reported 0.0088). Without area 25 the
sham-side LIM decreases (P = 6.9e-04), so the sham-side reading fails too (1 of 3), and on both
sides 2 of 6 are decreases. The interaction is F(1,10972) = 7.36, P = 0.0067 (reported). The
criterion blocks the claim when its readings disagree. The legend's LIM voxel counts (438 sham, 383 control) are those with area 25 (428 and 375
without), as are the Fig. 5c counts (821, against 803), so the paper's n support reading (a); a
reader is asked whether reading (b) should stay. Under reading (a) alone the sham-side and
both-sides readings still disagree, so the block does not depend on it.

## ed10d-monkey-d-roi-changes

> Among vmFC, PCC, anterior-dorsal insula and dACC, monkey D showed a significant pre against
> post difference in SCC connectivity only for the sham-side vmFC and the control-side
> anterior-dorsal insula.

In [ ]:
fit_roiD = dbs.anova(fcD, "fc", ["time_condition", "side", "roi"], T(None, "time_condition", "side", "roi"))
print(f"error df {fit_roiD['dfe']}:", fit_roiD["table"].loc["time_condition x side"].to_dict())
pp_roiD = dbs.prepost_pairs(dbs.tukey_kramer(fit_roiD, ["time_condition", "side", "roi"], only_vary="time_condition"), "roi", "time_condition")
pp_roiD["name"], pp_roiD["side"] = pp_roiD.roi.map(LAB), pp_roiD.side.map({1: "sham", 2: "control"})
four = pp_roiD[pp_roiD.name.isin(["vmFC", "PCC", "Anterior-dorsal Insula", "dACC"])]
print(four.to_string(index=False))
print("significant (P < 0.05):", [f"{s} {n}" for s, n in zip(four[four.P < 0.05].side, four[four.P < 0.05].name)])
print("\nall 17 ROIs, monkey D:")
print(pp_roiD.to_string(index=False))

**Result: reproduced.** PrePost x side F(1,10106) = 4.58, P = 0.032 (reported). Of the eight
comparisons, exactly the sham-side vmFC (decrease, P = 8.0e-04, reported 7.83e-04) and the
control-side anterior-dorsal insula (increase, P = 1.1e-11) have P < 0.05; the others have
P = 0.31 or more.

## Tukey-Kramer P values against the reported ones

The P values the paper reports (claims file, from Supplementary Table 9 and the figure legends)
next to those computed here, with their ratio. SciPy's studentized range tail sets the
accuracy of the small ones.

In [ ]:
def p_of(df, **sel):
    m = np.ones(len(df), bool)
    for k, v in sel.items():
        m &= (df[k] == v).values
    assert m.sum() == 1, sel
    return float(df[m].P.iloc[0])

tk_vs_reported = pd.DataFrame([
    ("N, T stim dACC", p_of(pp_roi, side="stim", name="dACC"), 4.75e-04),
    ("N, T stim HIP", p_of(pp_roi, side="stim", name="HIP"), 1.19e-05),
    ("N, T stim AMY", p_of(pp_roi, side="stim", name="AMY"), 7.70e-06),
    ("N, T control SAN", p_of(pp_net["a"], side="control", network="SAN"), 6.20e-07),
    ("N, T control MT", p_of(pp_roi, side="control", name="MT"), 7.85e-07),
    ("N, T stim insula", p_of(pp_roi, side="stim", name="Anterior-dorsal Insula"), 4.42e-08),
    ("N, T stim PCC", p_of(pp_roi, side="stim", name="PCC"), 8.22e-12),
    ("N, T stim motor cortex", p_of(pp_roi, side="stim", name="Motor Cx"), 0.99991),
    ("N, T stim SMA", p_of(pp_roi, side="stim", name="SMA"), 0.99999),
    ("N, T stim MCC", p_of(pp_roi, side="stim", name="MCC"), 0.99183),
    ("D sham CB (FA)", p_of(pp_faD, side="sham", tract="CB"), 5.07e-09),
    ("D sham dACC-CB (FA)", p_of(pp_spD, side="sham", part="dACC"), 1.30e-08),
    ("D control CEN", p_of(pp_netD["a"], side="control", network="CEN"), 0.0088),
    ("D sham vmFC", p_of(pp_roiD, side="sham", name="vmFC"), 7.83e-04),
    ("D control insula", p_of(pp_roiD, side="control", name="Anterior-dorsal Insula"), 1.37e-12),
], columns=["comparison", "P_here", "P_reported"])
tk_vs_reported["ratio"] = tk_vs_reported.P_here / tk_vs_reported.P_reported
print(tk_vs_reported.sort_values("P_reported", ascending=False).to_string(index=False, float_format=lambda x: f"{x:.3g}"))

## Summary

| Claim | Outcome |
|---|---|
| ed1-behavior-increase | reproduced |
| ed1-monkey-d-no-change | not reproduced |
| fig2a-cb-fa-increase | reproduced |
| fig2a-no-change-other-tracts | reproduced |
| ed2-cb-both-monkeys | reproduced |
| fig2b-mcc-cb-fa-increase | reproduced |
| fig3c-cc1-dapi-ratio | reproduced |
| ed5-cc1-count-dapi-unchanged | reproduced |
| ed6-dacc-ufm-no-oligodendrocyte-change | reproduced |
| fig3f-g-ratio-decrease | reproduced |
| fig4a-cortical-down-subcortical-up | reproduced |
| fig4b-roi-changes-counts | reproduced |
| fig4c-network-sorted-reduction | blocked (methods underspecified) |
| fig4-nonsurgical-no-pattern | reproduced |
| fig5a-pre-scc-connectivity | reproduced |
| fig5b-scc-decreases | reproduced |
| fig5b-scc-increases | reproduced |
| ed9-control-hemisphere-changes | reproduced |
| ed9-no-change-motor-mcc | reproduced |
| ed7-decreases-both-monkeys | reproduced |
| fig5a-nonsurgical-no-reduction | blocked (data missing) |
| fig5c-dmn-lim-cen-decrease | reproduced |
| fig5c-smn-increase | reproduced |
| fig5c-vis-no-change | reproduced |
| fig5c-san-rebalanced | reproduced |
| fig6a-monkey-d-cb-fa-decrease | reproduced |
| fig6b-monkey-d-dacc-cb-decrease | reproduced |
| fig6c-monkey-d-cc1-no-change | reproduced |
| ed10c-monkey-d-subcortical-decrease | reproduced |
| fig6d-monkey-d-scc-local-increase | blocked (data missing) |
| fig6e-monkey-d-no-network-reduction | blocked (methods underspecified) |
| ed10d-monkey-d-roi-changes | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The tests are those of the paper's MATLAB scripts
in the dandiset, reimplemented in Python: type III ANOVA with sum-to-zero coding and the scripts'
model terms, Tukey-Kramer on population marginal means, MATLAB's tie-corrected rank-sum. The
ANOVAs give the reported F values to two decimals throughout. The Tukey-Kramer P values are within 3% of
the reported ones above 1e-4, 1.12 to 1.14 times them near 1e-5 and 1.11 to 1.28 times near
1e-6 (table before this summary). Below 1e-7 SciPy's studentized range tail is inexact: 1.5 to
2.8 times the reported value between 5e-9 and 4.4e-8, 5.5 and 8.3 times for the reported 8.22e-12
and 1.37e-12, and it stops near 1e-11 or at 0 where the paper prints 0. No such P is near 0.05. The deposited seed maps are one
per session, and the paper does not say whether its runs were concatenated or averaged, so only
one of the claims file's two run readings can be run; the seed connectivity outcomes assume the
deposited maps are those the paper analysed, which the reported F values support. The per-ROI connectome counts use the test and
threshold of the paper's Fig. 4 script (rank-sum of the changes against zeros, P < 0.05/123), as
the claims file has them since a reader's answer; they differ from the Methods' "P < 0.01 with
Bonferroni correction".

**What was found about the dataset.** It holds derived values, not images, and with the paper's
scripts it supports almost every statistic of the DWI, histology, EM, behavior and seed
connectivity analyses. Missing: the nonsurgical controls' seed connectivity, all whole-brain
voxel maps, per-monkey connectomes, ROI names for the 123 x 123 connectomes, and the lead
location. Errors: the behavior legend codes foraging as yes 2 and no 1 while the values are 0 and
1, and gives no group code for monkey D (its rows carry 2); two MCC-CB fields of monkey T carry
ratios that do not match their counts, and two MCC-CB fields have more CC1-positive than
DAPI-positive cells; the EM column headed "diameter of axon" holds half the mean diameter; two
the FigE1 script computes the foraging standard errors from the movement column; two
scripts are deposited twice under different names. The connectomes have 123 ROIs, so each ROI has
122 values, the n of the Fig. 4b legend.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")